# Preprocessing
Starter notebook. Implementation pending.

In [1]:
from pathlib import Path
import re
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

source = Path(r"D:\ResistMap-AI\data\raw\isolates_with_metadata.tsv")
output = Path(r"D:\ResistMap-AI\data\processed\klebsiella_amr_features.csv")

df = pd.read_csv(source, sep="\t", dtype="string")

required = ["Scientific name", "Isolate", "AMR genotypes",
            "Collection date", "Location"]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

# Keep the exact species and one record per isolate.
df = df[df["Scientific name"].str.strip().eq("Klebsiella pneumoniae")].copy()
df = df.drop_duplicates(subset="Isolate")

def complete_genes(value):
    genes = set()
    if pd.isna(value):
        return []
    for item in str(value).split(","):
        if "=" not in item:
            continue
        gene, status = item.rsplit("=", 1)
        if status.strip() == "COMPLETE" and gene.strip():
            genes.add(gene.strip())
    return sorted(genes)

df["complete_genes"] = df["AMR genotypes"].map(complete_genes)
df = df[df["complete_genes"].map(len).gt(0)].copy()

# Dates can be YYYY, YYYY-MM, or YYYY-MM-DD. Extract only a valid year.
year_text = df["Collection date"].str.extract(r"^(\d{4})(?:-|$)")[0]
df["collection_year"] = pd.to_numeric(year_text, errors="coerce")
df.loc[~df["collection_year"].between(1900, 2026), "collection_year"] = pd.NA

# Reproducible 3,000-isolate prototype sample.
sample = df.sample(n=min(3000, len(df)), random_state=42).copy()
sample = sample.reset_index(drop=True)

encoder = MultiLabelBinarizer()
gene_array = encoder.fit_transform(sample["complete_genes"])
gene_columns = [f"gene__{gene}" for gene in encoder.classes_]

genes = pd.DataFrame(gene_array, columns=gene_columns, dtype="int8")
metadata = sample[["Isolate", "Scientific name", "Collection date",
                   "collection_year", "Location"]].reset_index(drop=True)

result = pd.concat([metadata, genes], axis=1)
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False)

print("Source rows:", len(df))
print("Saved sample rows:", len(result))
print("Gene feature columns:", len(gene_columns))
print("Missing collection year:", result["collection_year"].isna().sum())
print("Missing location:", result["Location"].isna().sum())
print("Output:", output)
display(result.iloc[:5, :10])

Source rows: 161452
Saved sample rows: 3000
Gene feature columns: 432
Missing collection year: 533
Missing location: 326
Output: D:\ResistMap-AI\data\processed\klebsiella_amr_features.csv


,Isolate,Scientific name,Collection date,collection_year,Location,gene__aac(3)-IId,gene__aac(3)-IIe,gene__aac(3)-IIg,gene__aac(3)-IVa,gene__aac(3)-Ia
0,PDT003244486.1,Klebsiella pneumoniae,2022,2022,China: Beijing,0,0,0,0,0
1,PDT000614742.1,Klebsiella pneumoniae,2017-12-18,2017,Italy,0,0,0,0,0
2,PDT003123211.1,Klebsiella pneumoniae,<NA>,<NA>,China: Beijing,0,0,0,0,0
3,PDT003030633.1,Klebsiella pneumoniae,2024,2024,Brazil:CAMPINAS,0,0,0,0,0
4,PDT002900855.1,Klebsiella pneumoniae,2025,2025,USA,0,0,0,0,0


In [2]:
gene_cols = [col for col in result.columns if col.startswith("gene__")]

print("Total gene columns:", len(gene_cols))
print("Total 1 values:", int(result[gene_cols].to_numpy().sum()))
print("Rows with zero COMPLETE genes:",
      int(result[gene_cols].sum(axis=1).eq(0).sum()))

print("\nFirst 5 samples: number of detected COMPLETE genes")
print(result[gene_cols].sum(axis=1).head())

print("\nMost common genes:")
print(result[gene_cols].sum().sort_values(ascending=False).head(10))

Total gene columns: 432
Total 1 values: 40922
Rows with zero COMPLETE genes: 0

First 5 samples: number of detected COMPLETE genes
0    10
1     5
2    15
3    12
4    22
dtype: int64

Most common genes:
gene__emrD           2982
gene__fosA           2861
gene__oqxA           2444
gene__blaTEM-1       1599
gene__oqxB           1483
gene__blaSHV-11      1351
gene__sul1           1261
gene__blaCTX-M-15    1225
gene__sul2           1125
gene__aph(6)-Id       957
dtype: int64


In [3]:
frequency = result[gene_cols].mean()
useful_genes = frequency[
    (frequency >= 0.01) & (frequency <= 0.99)
].index.tolist()

print("Original genes:", len(gene_cols))
print("Genes present in 1%–99% of samples:", len(useful_genes))
print("Excluded very rare genes:", int((frequency < 0.01).sum()))
print("Excluded nearly universal genes:", int((frequency > 0.99).sum()))

Original genes: 432
Genes present in 1%–99% of samples: 95
Excluded very rare genes: 336
Excluded nearly universal genes: 1
